# 个人作业2：桌游复杂度与玩家评分的量化关系（深度建模）
### 大数据处理技术课程 — 2025-2026第二学期

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

print('环境就绪')

In [ ]:
# 数据加载与清洗
df = pd.read_csv('data/bgg_board_games.csv')
rating_col = 'average_rating'

df = df[df['users_rated'] >= 30]
df = df[df['year_published'] > 0]
df = df[df['year_published'] <= 2026]
df = df[df['playing_time'] > 0]
df = df[(df[rating_col] >= 1) & (df[rating_col] <= 10)]

for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('Unknown')

df['log_users_rated'] = np.log1p(df['users_rated'])
df['mechanics_count'] = df['mechanic'].fillna('').str.split(',').apply(
    lambda x: len([i for i in x if i.strip() != ''])
)

time_bins = [0, 30, 60, 120, 99999]
time_labels = ['<30 min', '30-60 min', '60-120 min', '>120 min']
df['time_bin'] = pd.cut(df['playing_time'], bins=time_bins, labels=time_labels)

print(f'数据加载完成: {len(df):,} 款游戏, {df.shape[1]} 列')
print(f'年份: {df["year_published"].min():.0f}-{df["year_published"].max():.0f}')

In [ ]:
# 机制独热编码
all_mechanics = df['mechanic'].dropna().str.split(',').explode().str.strip()
mech_freq = all_mechanics.value_counts()
common_mechanics = mech_freq[mech_freq >= 100].index.tolist()

for mech in common_mechanics[:50]:
    col_name = f'mech__{mech.replace(" ", "_").replace("/", "_")}'
    df[col_name] = df['mechanic'].fillna('').str.contains(mech, regex=False).astype(int)

all_categories = df['category'].dropna().str.split(',').explode().str.strip()
cat_freq = all_categories.value_counts()
common_categories = cat_freq[cat_freq >= 200].index.tolist()[:20]
for cat in common_categories:
    col_name = f'cat__{cat.replace(" ", "_").replace("/", "_")}'
    df[col_name] = df['category'].fillna('').str.contains(cat, regex=False).astype(int)

feat_cols = [c for c in df.columns if c.startswith('mech__') or c.startswith('cat__')]
print(f'独热编码特征: {len(feat_cols)}')

## 一、PCA降维

In [ ]:
# PCA降维与可视化
X = df[feat_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)
explained_var = pca.explained_variance_ratio_
cumsum_var = np.cumsum(explained_var)

df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 碎石图
axes[0].bar(range(1, 16), explained_var[:15], alpha=0.7, color='steelblue')
axes[0].plot(range(1, 16), cumsum_var[:15], 'r-o', linewidth=2, markersize=5)
axes[0].set_xlabel('主成分')
axes[0].set_ylabel('方差解释率')
axes[0].set_title('PCA碎石图', fontsize=13, fontweight='bold')
axes[0].axhline(y=0.8, color='green', linestyle='--')

# PCA空间散点图
sc = axes[1].scatter(df['PC1'], df['PC2'], c=df[rating_col], cmap='RdYlGn', alpha=0.3, s=8)
axes[1].set_xlabel(f'PC1 ({explained_var[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({explained_var[1]:.1%})')
axes[1].set_title('PCA机制空间（按评分着色）', fontsize=13, fontweight='bold')
plt.colorbar(sc, ax=axes[1], label='评分')

plt.tight_layout()
plt.savefig('output/pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'PC1方差: {explained_var[0]:.1%}')
print(f'PC2方差: {explained_var[1]:.1%}')
print(f'PC1+2累计: {cumsum_var[1]:.1%}')
print(f'达到80%需主成分数: {np.argmax(cumsum_var >= 0.8) + 1}')

In [ ]:
# PCA载荷分析
loadings = pd.DataFrame(pca.components_[:2].T, index=feat_cols, columns=['PC1', 'PC2'])

print('PC1 正方向:')
for idx, row in loadings.nlargest(5, 'PC1').iterrows():
    name = idx.replace('mech__', '').replace('_', ' ')
    print(f'  {name}: +{row["PC1"]:.3f}')

print('\nPC1 负方向:')
for idx, row in loadings.nsmallest(5, 'PC1').iterrows():
    name = idx.replace('mech__', '').replace('_', ' ')
    print(f'  {name}: {row["PC1"]:.3f}')

print('\nPC1大致代表 策略向 vs 休闲向 这个轴')

## 二、K-Means聚类

In [ ]:
# K-Means聚类
cluster_features = ['PC1', 'PC2', 'mechanics_count', 'log_users_rated']
X_cluster = StandardScaler().fit_transform(df[cluster_features].dropna())

# 肘部法则
inertias = []
silhouettes = []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_cluster)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_cluster, labels))

best_k = K_range[np.argmax(silhouettes)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('K')
axes[0].set_ylabel('惯性')
axes[0].set_title('肘部法则', fontsize=13, fontweight='bold')
axes[0].axvline(x=best_k, color='red', linestyle='--', label=f'最优K={best_k}')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('K')
axes[1].set_ylabel('轮廓系数')
axes[1].set_title('轮廓系数 vs K', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('output/kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'最优K = {best_k}, 轮廓系数 = {max(silhouettes):.3f}')

In [ ]:
# 执行聚类并可视化
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster)

# 聚类画像
profile = df.groupby('cluster').agg(
    avg_rating=('average_rating', 'mean'),
    count=('average_rating', 'count'),
    med_time=('playing_time', 'median'),
    avg_mechanics=('mechanics_count', 'mean')
).round(2)
profile['pct'] = (profile['count'] / len(df) * 100).round(1)
print(profile)

# PCA空间中可视化聚类
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(df['PC1'], df['PC2'], c=df['cluster'], cmap='tab10', alpha=0.3, s=8)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title(f'K-Means聚类 (K={k})', fontsize=13, fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='聚类')

df['cluster'].value_counts().sort_index().plot(kind='pie', autopct='%1.1f%%', ax=axes[1])
axes[1].set_title('聚类分布', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('output/cluster_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 三、时间序列分析

In [ ]:
# 年度时间序列
yearly = df.groupby('year_published').agg(
    avg_rating=('average_rating', 'mean'),
    pub_count=('average_rating', 'count'),
    med_time=('playing_time', 'median'),
    avg_mechanics=('mechanics_count', 'mean')
).reset_index()
yearly = yearly[yearly['pub_count'] >= 20]
yearly = yearly[yearly['year_published'] <= 2020]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].fill_between(yearly['year_published'], yearly['pub_count'], alpha=0.3, color='steelblue')
axes[0, 0].plot(yearly['year_published'], yearly['pub_count'], 'b-', linewidth=1.5)
axes[0, 0].set_title('年度发行量', fontsize=13, fontweight='bold')

axes[0, 1].plot(yearly['year_published'], yearly['avg_rating'], 'o-', markersize=3, color='coral', linewidth=1)
axes[0, 1].set_title('平均评分趋势', fontsize=13, fontweight='bold')

axes[1, 0].bar(yearly['year_published'], yearly['avg_mechanics'], color='steelblue', alpha=0.7)
axes[1, 0].set_title('平均机制数量趋势', fontsize=13, fontweight='bold')

axes[1, 1].plot(yearly['year_published'], yearly['med_time'], 'g-', linewidth=2)
axes[1, 1].set_title('中位游戏时长趋势', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('output/time_series.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'时间序列: {len(yearly)} 年')

In [ ]:
# 年代对比
pre95 = yearly[yearly['year_published'] <= 1995]
post00 = yearly[yearly['year_published'] >= 2000]
post15 = yearly[yearly['year_published'] >= 2015]

print(f'{"指标":<20} {"1995年前":>10} {"2000年后":>10} {"2015年后":>10}')
print('-' * 50)
print(f'{"年均出版量":<20} {pre95["pub_count"].mean():>10.0f} {post00["pub_count"].mean():>10.0f} {post15["pub_count"].mean():>10.0f}')
print(f'{"平均评分":<20} {pre95["avg_rating"].mean():>10.2f} {post00["avg_rating"].mean():>10.2f} {post15["avg_rating"].mean():>10.2f}')
print(f'{"平均机制数":<20} {pre95["avg_mechanics"].mean():>10.1f} {post00["avg_mechanics"].mean():>10.1f} {post15["avg_mechanics"].mean():>10.1f}')

## 综合结论

In [ ]:
print(f'PCA: 将{len(feat_cols)}个特征压到2个主成分，解释了{cumsum_var[1]:.1%}的方差')
print('  PC1基本就是策略向 vs 休闲向的轴\n')

print(f'K-Means: 找到{k}个桌游类型（最优K={best_k}）')
for c in range(k):
    cp = profile.loc[c]
    print(f'  第{c}类: {cp["count"]:,}款({cp["pct"]}%), 均分{cp["avg_rating"]:.2f}, 中位时长{cp["med_time"]:.0f}min')

print(f'\n时间序列: 2000年后桌游出版量明显增长')
print(f'  出版量: {post15["pub_count"].mean() / max(pre95["pub_count"].mean(), 1):.0f}倍')
print(f'  评分变化: {post15["avg_rating"].mean() - pre95["avg_rating"].mean():+.2f}')

print(f'\n游戏时长和评分基本不相关——不管是简单还是复杂的游戏，评分高的都有。')